# Compare Processed vs Raw Datasets
This notebook loads the processed and raw dataset versions, compares schemas and summaries, and visualizes differences using matplotlib.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [ ]:
# File paths (workspace-relative)
base = Path('data')
processed_path = base / 'processed' / 'spotify_clean.csv'
raw_path = base / 'raw' / 'songs_with_attributes_and_lyrics.csv'
print('Processed path:', processed_path)
print('Raw path:', raw_path)

In [ ]:
# Load datasets with relative-path diagnostics and fallback resolution
candidate_bases = [
    base,
    Path('./data'),
    Path('../data')
]

# remove duplicates while preserving order
seen = set()
candidate_bases = [p for p in candidate_bases if not (str(p) in seen or seen.add(str(p)))]

def resolve_file(subdir, filename):
    for b in candidate_bases:
        p = b / subdir / filename
        if p.exists():
            return p
    return candidate_bases[0] / subdir / filename

processed_path = resolve_file('processed', 'spotify_clean.csv')
raw_path = resolve_file('raw', 'songs_with_attributes_and_lyrics.csv')

print('Current working directory:', Path.cwd())
print('Resolved processed path:', processed_path)
print('Resolved raw path:', raw_path)

def safe_read_csv(p):
    if not p.exists():
        print(f'Missing file: {p}')
        return None
    try:
        return pd.read_csv(p, low_memory=False)
    except Exception as e:
        print(f'Error reading {p}:', e)
        return None

df_proc = safe_read_csv(processed_path)
df_raw = safe_read_csv(raw_path)

# Show basic previews
for name, df in [('processed', df_proc), ('raw', df_raw)]:
    print('-----', name, '-----')
    if df is None:
        print('Not available')
    else:
        print('shape:', df.shape)
        display(df.head())


In [ ]:
# Compare columns
if df_proc is not None and df_raw is not None:
    cols_proc = set(df_proc.columns)
    cols_raw = set(df_raw.columns)
    print('Columns only in processed:', sorted(cols_proc - cols_raw))
    print('Columns only in raw:', sorted(cols_raw - cols_proc))
    print('Common columns (sample):', sorted(list(cols_proc & cols_raw))[:20])
else:
    print('One or both dataframes missing; skipping column comparison')

In [ ]:
# Summary statistics for numeric columns and a combined comparison table
def numeric_summary(df):
    if df is None: return pd.DataFrame()
    nums = df.select_dtypes(include=[np.number])
    return nums.describe().T

summary_proc = numeric_summary(df_proc)
summary_raw = numeric_summary(df_raw)

print('Processed numeric summary (top rows)')
display(summary_proc.head())
print('Raw numeric summary (top rows)')
display(summary_raw.head())

# Build a delta table for shared numeric columns
if not summary_proc.empty and not summary_raw.empty:
    common_nums = set(summary_proc.index) & set(summary_raw.index)
    rows = []
    for col in sorted(common_nums):
        a = summary_proc.loc[col] if col in summary_proc.index else None
        b = summary_raw.loc[col] if col in summary_raw.index else None
        rows.append({
            'column': col,
            'proc_count': a['count'] if a is not None else np.nan,
            'raw_count': b['count'] if b is not None else np.nan,
            'proc_mean': a['mean'] if a is not None else np.nan,
            'raw_mean': b['mean'] if b is not None else np.nan,
            'mean_delta': (a['mean'] - b['mean']) if (a is not None and b is not None) else np.nan
        })
    delta_df = pd.DataFrame(rows).set_index('column')
    display(delta_df.sort_values('mean_delta', key=abs, ascending=False).head(20))
else:
    print('Insufficient numeric summaries to compute deltas')

In [ ]:
# Missing value comparison (counts and percent)
def missing_table(df):
    if df is None: return pd.DataFrame()
    mcount = df.isna().sum()
    mperc = (mcount / len(df)) * 100
    return pd.DataFrame({'missing_count': mcount, 'missing_percent': mperc})

miss_proc = missing_table(df_proc)
miss_raw = missing_table(df_raw)

print('Processed missing (top)')
if not miss_proc.empty and 'missing_percent' in miss_proc.columns:
    display(miss_proc.sort_values('missing_percent', ascending=False).head())
elif not miss_proc.empty:
    display(miss_proc.head())
else:
    print('No missing-value info for processed dataset')
print('Raw missing (top)')
if not miss_raw.empty and 'missing_percent' in miss_raw.columns:
    display(miss_raw.sort_values('missing_percent', ascending=False).head())
elif not miss_raw.empty:
    display(miss_raw.head())
else:
    print('No missing-value info for raw dataset')

In [ ]:
# Record-level matching: robust chunked comparison and diagnostics
# Print first 5 rows explicitly to show what's in each CSV
print('First 5 rows of processed (if loaded):')
if df_proc is None:
    print(' processed dataset not loaded')
else:
    display(df_proc.head(5))

print('First 5 rows of raw (if loaded):')
if df_raw is None:
    print(' raw dataset not loaded')
else:
    display(df_raw.head(5))

# helper: simple normalization for keys (lower, remove punctuation, collapse spaces)
import re
def norm_key_str(s):
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    s = re.sub(r'+', ' ', s).strip()
    return s

def make_key_from_cols(df, cols):
    # returns Series of keys; missing columns result in empty string parts
    parts = []
    for c in cols:
        if c in df.columns:
            parts.append(df[c].fillna('').map(norm_key_str))
        else:
            parts.append(pd.Series(['']*len(df), index=df.index))
    return parts[0].astype(str) + '||' + parts[1].astype(str) if len(parts) >= 2 else parts[0].astype(str)

# Build processed key set from normalized columns if present, else from title/artist
proc_key_cols = None
if df_proc is not None:
    if 'normalized_title' in df_proc.columns and 'normalized_artist' in df_proc.columns:
        proc_key_cols = ('normalized_title', 'normalized_artist')
    elif 'title' in df_proc.columns and 'artist' in df_proc.columns:
        proc_key_cols = ('title', 'artist')

proc_keys = set()
if proc_key_cols is not None:
    proc_keys = set(make_key_from_cols(df_proc, proc_key_cols).astype(str).tolist())
    print(f'Processed key count: {len(proc_keys)}')
else:
    print('No suitable key columns found in processed dataset')

# If processed keys exist, perform chunked comparison against raw CSV to avoid huge merges
match_count = 0
raw_only_count = 0
proc_only_count = 0
sample_matches = []
sample_raw_only = []

if proc_keys and raw_path.exists():
    chunksize = 50000
    for i, chunk in enumerate(pd.read_csv(raw_path, chunksize=chunksize, low_memory=False)):
        # prefer raw name columns mapping
        if 'name' in chunk.columns and 'artists' in chunk.columns:
            raw_cols = ('name', 'artists')
        elif 'title' in chunk.columns and 'artist' in chunk.columns:
            raw_cols = ('title', 'artist')
        elif 'track_name' in chunk.columns and 'artist' in chunk.columns:
            raw_cols = ('track_name','artist')
        else:
            # build a best-effort key from available first two text columns
            text_cols = [c for c in chunk.columns if chunk[c].dtype == object][:2]
            raw_cols = tuple(text_cols) if len(text_cols) >= 2 else (text_cols[0],) if text_cols else ()
        if not raw_cols:
            continue
        rk = make_key_from_cols(chunk, raw_cols).astype(str).tolist()
        # compute matches
        for j, k in enumerate(rk):
            if k in proc_keys:
                match_count += 1
                if len(sample_matches) < 10:
                    sample_matches.append(chunk.iloc[j].to_dict())
            else:
                raw_only_count += 1
                if len(sample_raw_only) < 10:
                    sample_raw_only.append(chunk.iloc[j].to_dict())
        print(f'Processed raw chunk {i}: matches so far {match_count}')
    # processed-only count = processed rows - matches (approx)
    proc_only_count = max(0, len(proc_keys) - match_count)
    print('Match summary: matched:', match_count, 'raw-only:', raw_only_count, 'proc-only (approx):', proc_only_count)
    print('Sample matched raw rows:')
    display(pd.DataFrame(sample_matches).head())
    print('Sample raw-only rows:')
    display(pd.DataFrame(sample_raw_only).head())
else:
    print('Skipping chunked comparison: either processed keys are missing or raw file not found')

In [ ]:
# Prepare output directory for figures and reports
out_dir = Path('outputs') / 'compare_report'
out_dir.mkdir(parents=True, exist_ok=True)
print('Output directory:', out_dir.resolve())

In [ ]:
# Visualizations
# 1) Simple bar chart for row/column counts
labels = ['processed','raw']
row_counts = [len(df_proc) if df_proc is not None else 0, len(df_raw) if df_raw is not None else 0]
col_counts = [df_proc.shape[1] if df_proc is not None else 0, df_raw.shape[1] if df_raw is not None else 0]
fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].bar(labels, row_counts, color=['C0','C1'])
axes[0].set_title('Row counts')
axes[1].bar(labels, col_counts, color=['C0','C1'])
axes[1].set_title('Column counts')
fig.tight_layout()
fig.savefig(out_dir / 'counts.png')
plt.show()

# 2) Overlaid histograms for selected numeric columns (if present)
candidate_numeric = ['duration_ms','tempo','danceability','energy','popularity']
if df_proc is not None and df_raw is not None:
    common_numeric = [c for c in candidate_numeric if c in df_proc.columns and c in df_raw.columns]
else:
    common_numeric = []

print('Numeric columns to compare:', common_numeric)
for col in common_numeric[:4]:  # limit to first 4 to keep notebook concise
    a = pd.to_numeric(df_proc[col], errors='coerce').dropna()
    b = pd.to_numeric(df_raw[col], errors='coerce').dropna()
    if len(a)==0 or len(b)==0:
        print(f'Skipping {col}: insufficient data')
        continue
    rng = np.nanpercentile(np.concatenate([a.values, b.values]), [1,99])
    plt.figure(figsize=(6,3))
    bins = 50
    plt.hist(a, bins=bins, alpha=0.6, label='processed', range=rng, color='C0')
    plt.hist(b, bins=bins, alpha=0.6, label='raw', range=rng, color='C1')
    plt.title(f'Distribution of {col}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f'hist_{col}.png')
    plt.show()

# 3) Scatter plot for paired records for a chosen numeric column (popularity if available)
if 'merged' in globals() and merged is not None and '_merge' in merged.columns and 'popularity_proc' in merged.columns and 'popularity_raw' in merged.columns:
    paired = merged[merged['_merge']=='both']
    x = pd.to_numeric(paired['popularity_proc'], errors='coerce')
    y = pd.to_numeric(paired['popularity_raw'], errors='coerce')
    plt.figure(figsize=(5,5))
    plt.scatter(x, y, alpha=0.4, s=10)
    plt.xlabel('popularity (processed)')
    plt.ylabel('popularity (raw)')
    plt.title('Popularity: processed vs raw (matched records)')
    lim = np.nanmax([x.max(), y.max()]) if (len(x.dropna())>0 and len(y.dropna())>0) else None
    if lim is not None and not np.isnan(lim):
        plt.plot([0,lim],[0,lim], color='k', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.savefig(out_dir / 'scatter_popularity.png')
    plt.show()
else:
    print('Skipping paired scatter: required popularity_proc/popularity_raw or matched records not available')

# 4) Top artists / categories comparison (if artist present)
artist_col = None
for c in ['artist','artists','artist_name']:
    if df_proc is not None and c in df_proc.columns:
        artist_col = c; break

if artist_col and df_raw is not None and artist_col in df_raw.columns:
    top_proc = df_proc[artist_col].value_counts().head(10)
    top_raw = df_raw[artist_col].value_counts().head(10)
    combined = pd.DataFrame({'processed': top_proc, 'raw': top_raw}).fillna(0)
    combined.plot(kind='bar', figsize=(10,4))
    plt.title('Top artists: processed vs raw (top 10)')
    plt.tight_layout()
    plt.savefig(out_dir / 'top_artists.png')
    plt.show()
else:
    print('Artist column not found in both datasets; skipping top-artist chart')

In [ ]:
# Save summary tables to disk
if 'delta_df' in globals():
    delta_df.to_csv(out_dir / 'numeric_deltas.csv')
if 'miss_proc' in globals():
    miss_proc.to_csv(out_dir / 'missing_processed.csv')
if 'miss_raw' in globals():
    miss_raw.to_csv(out_dir / 'missing_raw.csv')
print('Saved summary tables (if available) to', out_dir)

## Conclusions & Next Steps
- Inspect the saved figures in the outputs/compare_report folder.
- If there are large deltas for important numeric fields, consider row-level checks and provenance tracing.
- Optionally extend the notebook with time-series comparisons or additional columns of interest.